# Day 4 Practice: Advanced Golden Features & Automated Selection

**Role:** Competitive Kaggle Grandmaster

Welcome to the elite level of feature engineering. Today, we move beyond simple scaling and categorical maps. You will implement high-performance, leakage-proof transformations and automated pruning loops to handle high-dimensional, complex relational data.


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
import sys
import os

# Setup validation
try:
    import ipytest
except ImportError:
    !pip install -q ipytest
    import ipytest

if not os.path.exists('validate_day4.py'):
    print('⚠️ validate_day4.py not found!')
else:
    import validate_day4

ipytest.autoconfig()


## 1. Synthetic Relational Dataset
Run the cell below to generate our competition-style dataset. It contains user transaction sequences, high-cardinality categories, and collinear noise.


In [ ]:
np.random.seed(42)
n_users = 1000
rows_per_user = 5
n_rows = n_users * rows_per_user

df = pd.DataFrame({
    'User_ID': np.repeat(np.arange(n_users), rows_per_user),
    'High_Card_Cat': np.random.choice([f'Cat_{i}' for i in range(250)], n_rows),
    'Spend': np.random.gamma(2, 50, n_rows),
    'Transaction_Seq': np.tile(np.arange(rows_per_user), n_users),
    'Target': np.random.randint(0, 2, n_rows)
})

# Adding intentional collinearity (> 0.92 correlation)
df['Collinear_Feature'] = df['Spend'] * 1.5 + np.random.normal(0, 0.5, n_rows)
df['Random_Noise'] = np.random.randn(n_rows)

print(f"Dataset Shape: {df.shape}")
df.head()


## Challenge 1: Leakage-Proof Custom Target Encoder
**Task:** Implement a Target Encoder from scratch. 
1. Use **5-Fold Out-of-Fold (OOF)** cross-validation to calculate means (i.e., the mean for a row in Fold 1 must be calculated using only Folds 2-5).
2. Apply **Smoothing** with $m=10$ to handle rare categories.
3. Store result in `Category_Target`.


In [ ]:
# YOUR CODE HERE
# 1. Init Category_Target column
# 2. Setup KFold(n_splits=5)
# 3. Loop: Calc smoothed means on Train-folds, map to Val-fold
# Formula: (mean * count + global_mean * m) / (count + m)


In [ ]:
%%ipytest -qq
try:
    validate_day4.test_task_1()
except NameError: pass


## Challenge 2: Contextual Behavioral Features
**Task:** Use Pandas `.transform()` to create 3 features that give the model context about a user's spending habits:
1. `User_Mean_Spend`: Average spend per `User_ID`.
2. `User_Spend_Std`: Standard deviation of spend per `User_ID`.
3. `Spend_Rel_to_User`: Current `Spend` minus `User_Mean_Spend`.


In [ ]:
# YOUR CODE HERE



In [ ]:
%%ipytest -qq
try:
    validate_day4.test_task_2()
except NameError: pass


## Challenge 3: Automated Collinear Pruning
**Task:** Write a loop that inspects the correlation matrix. Identify and store the names of all columns that have a correlation $> 0.92$ with *any other column already in the set*. Store these names in a list called `to_drop`.


In [ ]:
# YOUR CODE HERE
# Hint: Use df.corr().abs() and np.triu(..., k=1)


In [ ]:
%%ipytest -qq
try:
    validate_day4.test_task_3()
except NameError: pass


## Challenge 4: Feature Importance Ranking
**Task:** Use a default `LGBMClassifier` to rank all features (excluding the `Target` and `User_ID`). Store the resulting importances in a Series named `feature_importances`.


In [ ]:
# YOUR CODE HERE



In [ ]:
%%ipytest -qq
try:
    validate_day4.test_task_4()
except NameError: pass


<details>
<summary style='color: #d35400; font-weight: bold; cursor: pointer;'>🏆 Click for Grandmaster Optimized Solutions</summary>

```python
# Solution 1: OOF Target Encoding
m = 10
global_mean = df['Target'].mean()
df['Category_Target'] = 0.0
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(df):
    train_fold = df.iloc[train_idx]
    agg = train_fold.groupby('High_Card_Cat')['Target'].agg(['count', 'mean'])
    smoothed = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    df.loc[df.index[val_idx], 'Category_Target'] = df.loc[df.index[val_idx], 'High_Card_Cat'].map(smoothed).fillna(global_mean)

# Solution 2: Contextual Features
df['User_Mean_Spend'] = df.groupby('User_ID')['Spend'].transform('mean')
df['User_Spend_Std'] = df.groupby('User_ID')['Spend'].transform('std')
df['Spend_Rel_to_User'] = df['Spend'] - df['User_Mean_Spend']

# Solution 3: Collinear Pruning
corr_matrix = df.select_dtypes(include=[np.number]).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.92)]

# Solution 4: LightGBM Importance
X = df.select_dtypes(include=[np.number]).drop(columns=['Target', 'User_ID'])
y = df['Target']
model = lgb.LGBMClassifier(verbose=-1).fit(X, y)
feature_importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
```
</details>
